In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Build features

In [ ]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()

builder.svd_explained_variance

np.float64(0.3152933276314166)

Convert each anime in df to vectors

In [6]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [7]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Bayesion Ridge Regression

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.linear_model import BayesianRidge
import numpy as np

rated_items, anime_df, anime_vectors, anime_df_scaled, builder = anime_data_client.get_rated_items(
    user_scores=user_scores,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
    anime_df=anime_df,
    anime_vectors=anime_vectors,
)

X = np.array([vec for _, vec, _ in rated_items])
y = np.array([score for _, _, score in rated_items], dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=None
)

model = BayesianRidge()
model.fit(X_train, y_train)

pred, pred_std = model.predict(X_test, return_std=True)
pred = np.clip(pred, 1, 10)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)

baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Improvement vs baseline: {baseline_mae - mae:.3f}")

MAE: 0.989
RMSE: 1.313
Baseline MAE: 1.309
Improvement vs baseline: 0.320


In [9]:
print("num ratings:", len(y))
print("mean:", y.mean())
print("median:", np.median(y))
print("std:", y.std())
print(pd.Series(y).value_counts().sort_index())

num ratings: 260
mean: 7.403846153846154
median: 7.0
std: 1.5696596321963387
1.0      1
3.0      2
4.0      8
5.0     10
6.0     57
7.0     57
8.0     48
9.0     59
10.0    18
Name: count, dtype: int64


Hit Rate

In [10]:
# Hide some high-rated anime, train on the rest, and see if the hidden likes show up near the top.
hit_rating_threshold = np.median(y) + 0.5 * np.std(y)
heldout_fraction = 0.25
top_ks = [5, 10, 20, 50, 100]

rated_eval = pd.DataFrame({
    "anime_id": [anime_id for anime_id, _, _ in rated_items],
    "score": [score for _, _, score in rated_items],
})

liked_eval = rated_eval[rated_eval["score"] >= hit_rating_threshold]
if len(liked_eval) < 2:
    raise ValueError("Need at least 2 high-rated anime to run a hit-rate holdout test.")

heldout_liked = liked_eval.sample(frac=heldout_fraction, random_state=None)
heldout_ids = set(heldout_liked["anime_id"])

train_eval = rated_eval[~rated_eval["anime_id"].isin(heldout_ids)]
train_ids = train_eval["anime_id"].tolist()

X_hit_train = anime_df_scaled.loc[train_ids].to_numpy()
y_hit_train = train_eval["score"].to_numpy(dtype=float)

hit_model = BayesianRidge()
hit_model.fit(X_hit_train, y_hit_train)

# Candidates are everything the model did not train on, including the hidden liked anime.
candidate_ids = [anime_id for anime_id in anime_df_scaled.index if anime_id not in set(train_ids)]
X_hit_candidates = anime_df_scaled.loc[candidate_ids].to_numpy()

hit_pred, hit_pred_std = hit_model.predict(X_hit_candidates, return_std=True)
hit_pred = np.clip(hit_pred, 1, 10)

title_by_id = {int(anime["id"]): anime["title"] for anime in anime_data.values()}
hit_recommendations = pd.DataFrame({
    "anime_id": candidate_ids,
    "title": [title_by_id.get(int(anime_id), "Unknown") for anime_id in candidate_ids],
    "actual_score": [user_scores.get(int(anime_id)) for anime_id in candidate_ids],
    "predicted_score": hit_pred,
    "uncertainty_raw": hit_pred_std,
})

uncertainty_clipped = hit_recommendations["uncertainty_raw"].clip(
    lower=hit_recommendations["uncertainty_raw"].quantile(0.05),
    upper=hit_recommendations["uncertainty_raw"].quantile(0.95),
)
if uncertainty_clipped.max() == uncertainty_clipped.min():
    hit_recommendations["uncertainty"] = 0.0
else:
    hit_recommendations["uncertainty"] = (
        (uncertainty_clipped - uncertainty_clipped.min())
        / (uncertainty_clipped.max() - uncertainty_clipped.min())
    )

hit_recommendations["ranking_score"] = (
    hit_recommendations["predicted_score"] - 4.5 * hit_recommendations["uncertainty"]
)
hit_recommendations["is_hidden_like"] = hit_recommendations["anime_id"].isin(heldout_ids)
hit_recommendations = hit_recommendations.sort_values("ranking_score", ascending=False)

hit_rows = []
for k in top_ks:
    top_k = hit_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    hit_rows.append({
        "k": k,
        "hits": hits,
        "heldout_likes": len(heldout_ids),
        "hit_rate": hits / len(heldout_ids),
        "precision_at_k": hits / k,
    })

hit_rate_results = pd.DataFrame(hit_rows)

print(f"Total rated anime: {len(rated_eval)}")
print(f"High-rated anime (score >= {hit_rating_threshold}): {len(liked_eval)}")
print(f"Hidden liked anime: {len(heldout_ids)}")
display(hit_rate_results)

# hit_recommendations[hit_recommendations["is_hidden_like"]].head(20)

Total rated anime: 260
High-rated anime (score >= 7.78482981609817): 125
Hidden liked anime: 31


,k,hits,heldout_likes,hit_rate,precision_at_k
0,5,2,31,0.064516,0.40
1,10,3,31,0.096774,0.30
2,20,4,31,0.129032,0.20
3,50,8,31,0.258065,0.16
4,100,12,31,0.387097,0.12


In [11]:
baseline_recommendations = hit_recommendations.copy()

# If "mean" is available in anime_df, rank by global MAL mean.
baseline_recommendations["baseline_score"] = anime_df.loc[
    baseline_recommendations["anime_id"], "mean"
].to_numpy()

baseline_recommendations = baseline_recommendations.sort_values(
    "baseline_score",
    ascending=False
)

baseline_rows = []
for k in top_ks:
    top_k = baseline_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    baseline_rows.append({
        "k": k,
        "baseline_hits": hits,
        "heldout_likes": len(heldout_ids),
        "baseline_hit_rate": hits / len(heldout_ids),
        "baseline_precision_at_k": hits / k,
    })

baseline_hit_rate_results = pd.DataFrame(baseline_rows)
hit_rate_results.merge(baseline_hit_rate_results, on=["k", "heldout_likes"])

,k,hits,heldout_likes,hit_rate,precision_at_k,baseline_hits,baseline_hit_rate,baseline_precision_at_k
0,5,2,31,0.064516,0.40,1,0.032258,0.20
1,10,3,31,0.096774,0.30,1,0.032258,0.10
2,20,4,31,0.129032,0.20,3,0.096774,0.15
3,50,8,31,0.258065,0.16,3,0.096774,0.06
4,100,12,31,0.387097,0.12,5,0.161290,0.05


Tuning Bayesian

In [20]:
weights_uncertainty = np.array([
    0, 1, 2, 3,
    3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8,
    8.5, 9, 10, 11, 12
])
n_runs = 800
tuning_top_ks = [5, 10]

from anime_evaluation import HitRateEvaluator

hitman = HitRateEvaluator(
            anime_df_scaled=anime_df_scaled,
            anime_df=anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = hitman.tune_bayesian_uncertainty(
    weights=weights_uncertainty,
    n_runs=n_runs,
    top_ks=tuning_top_ks,
    random_state=42,
)

best_bayesian_weights = best_bayesian_weights.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

average_metrics = bayesian_summary.merge(
    baseline_summary,
    on="k",
    how="left",
).rename(columns={"uncertainty_weight": "bayesian_uncertainty_weight"})

best_bayesian_weights

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,7.5,5,0.522500,0.200609,0.084274,0.032356,2.61250,0.09450,0.120366,0.015242,0.019414,0.4725
1,3.5,10,0.341375,0.120805,0.110121,0.038969,3.41375,0.06325,0.058769,0.020403,0.018958,0.6325


In [21]:
bayesian_summary

,uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits
24,7.5,5,0.522500,0.200609,0.084274,0.032356,2.61250
26,8.0,5,0.519750,0.200899,0.083831,0.032403,2.59875
22,7.0,5,0.518750,0.201244,0.083669,0.032459,2.59375
20,6.5,5,0.514750,0.203308,0.083024,0.032792,2.57375
28,8.5,5,0.513750,0.199652,0.082863,0.032202,2.56875
30,9.0,5,0.509750,0.197113,0.082218,0.031792,2.54875
18,6.0,5,0.509000,0.203890,0.082097,0.032885,2.54500
16,5.5,5,0.497750,0.205786,0.080282,0.033191,2.48875
32,10.0,5,0.491750,0.196927,0.079315,0.031762,2.45875
14,5.0,5,0.485750,0.198106,0.078347,0.031953,2.42875


## Results

The notebook cell above uses the updated **800-run** tuning experiment for user `chekkit`; the final metrics are mirrored in `metrics/bayesian_uncertainty_tuning_20260615_153918.csv`.

| k | Best uncertainty weight | Avg precision@k | Std precision@k | Avg hit rate | Avg hits | Baseline precision@k |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 5 | 7.5 | 0.5225 | 0.2006 | 0.0843 | 2.6125 | 0.0945 |
| 10 | 3.5 | 0.3414 | 0.1208 | 0.1101 | 3.4138 | 0.0633 |

For `k=5`, the best result was at uncertainty weight **7.5**, with nearby weights from about **6.5 to 8.5** performing very similarly. For `k=10`, the best result was at uncertainty weight **3.5**, with nearby weights around **3.0 to 4.0** close behind.

Conclusion: use **7.5** when optimizing short top-5 recommendations, and use **3.5** when optimizing top-10 lists. If one default is needed, prefer **7.5** because Precision@5 is the more important cutoff for a short recommendation list.
